In [2]:
import os
import sys
import io
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy.stats import ttest_ind  # for statistical comparison

# ------------------------------------------------------------------------------
# SETUP: Create output folders for text and plots.
# ------------------------------------------------------------------------------
results_folder = "results"
plots_folder = os.path.join(results_folder, "plots")
text_folder = os.path.join(results_folder, "text")
corr_folder = os.path.join(plots_folder, "correlation")
boxplots_folder = os.path.join(plots_folder, "boxplots")

for folder in [results_folder, plots_folder, text_folder, corr_folder, boxplots_folder]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# ------------------------------------------------------------------------------
def analyze_llm_results(csv_path):
    """
    Reads the CSV file of experiment results, renames columns to valid identifiers,
    and performs the following:
      1. Data loading and cleaning
      2. Export descriptive stats to a text file
      3. Factor-level comparisons (ANOVA) with results exported to a text file
      4. Correlation analysis (text file + saved heatmap)
      5. Visualization (boxplots saved into metric-specific subfolders)
      6. Deepseek vs. Each Other Model Analysis (basic statistical comparison)
    """
    # 1. DATA LOADING & CLEANING
    df = pd.read_csv(csv_path, sep=",")
    
    # Trim whitespace from column names
    df.columns = [col.strip() for col in df.columns]
    
    # Rename columns to valid variable names (remove spaces, hyphens, and parentheses)
    rename_dict = {
        "Case study": "Case_study",
        "Summary": "Summary",
        "LLM": "LLM",
        "Role": "Role",
        "Example": "Example",
        "Insight": "Insight",
        "BLEU": "BLEU",
        "METEOR": "METEOR",
        "ROUGE-1": "ROUGE_1",
        "ROUGE-2": "ROUGE_2",
        "ROUGE-L": "ROUGE_L",
        "Flesch Reading Ease": "Flesch_Reading_Ease",
        "Coverage (Claude)": "Coverage_Claude",
        "Faithfulness (GPT)": "Faithfulness_GPT"
    }
    df.rename(columns=rename_dict, inplace=True)
    
    # Convert appropriate columns to categorical dtype
    categorical_cols = ["Case_study", "Summary", "LLM", "Role", "Example", "Insight"]
    for col in categorical_cols:
        df[col] = df[col].astype('category')
    
    # ------------------------------------------------------------------------------
    # 2. OUTPUT: Descriptive Statistics
    # ------------------------------------------------------------------------------
    desc_stats = df.describe().to_string()
    with open(os.path.join(text_folder, "descriptive_stats.txt"), "w") as f:
        f.write(desc_stats)
    
    # ------------------------------------------------------------------------------
    # 3. FACTOR-LEVEL COMPARISONS (ANOVA)
    # ------------------------------------------------------------------------------
    metrics = ["BLEU", "METEOR", "ROUGE_1", "ROUGE_2", "ROUGE_L", 
               "Flesch_Reading_Ease", "Coverage_Claude", "Faithfulness_GPT"]
    
    with open(os.path.join(text_folder, "anova_results.txt"), "w") as f:
        for m in metrics:
            try:
                formula = (f"{m} ~ C(Case_study) + C(Summary) + C(LLM) + "
                           f"C(Role) + C(Example) + C(Insight)")
                model = ols(formula, data=df).fit()
                anova_table = sm.stats.anova_lm(model, typ=2)
                f.write(f"ANOVA results for {m}:\n")
                f.write(anova_table.to_string())
                f.write("\n\n")
            except Exception as e:
                f.write(f"Could not run ANOVA for {m} due to error: {e}\n\n")
    
    # ------------------------------------------------------------------------------
    # 4. CORRELATION ANALYSIS
    # ------------------------------------------------------------------------------
    numeric_df = df[metrics].dropna()  # Drop rows with missing numeric values
    corr_matrix = numeric_df.corr()
    with open(os.path.join(text_folder, "correlation_matrix.txt"), "w") as f:
        f.write(corr_matrix.to_string())
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Correlation Matrix of Evaluation Metrics")
    plt.tight_layout()
    corr_plot_path = os.path.join(corr_folder, "correlation_matrix.png")
    plt.savefig(corr_plot_path)
    plt.close()
    
    # ------------------------------------------------------------------------------
    # 5. VISUALIZATION: Boxplots
    # ------------------------------------------------------------------------------
    # Create a dedicated subfolder for each metric inside the boxplots folder.
    for m in metrics:
        metric_folder = os.path.join(boxplots_folder, m)
        if not os.path.exists(metric_folder):
            os.makedirs(metric_folder)
        
        # Boxplot for each metric by LLM
        plt.figure(figsize=(8, 6))
        sns.boxplot(x="LLM", y=m, data=df)
        plt.title(f"{m} by LLM")
        plt.tight_layout()
        boxplot_path = os.path.join(metric_folder, f"boxplot_{m}_by_LLM.png")
        plt.savefig(boxplot_path)
        plt.close()
        
        # Boxplots for each metric by factors 'Role', 'Example', and 'Insight'
        for factor in ["Role", "Example", "Insight"]:
            plt.figure(figsize=(8, 6))
            sns.boxplot(x=factor, y=m, data=df)
            plt.title(f"{m} by {factor}")
            plt.tight_layout()
            boxplot_path = os.path.join(metric_folder, f"boxplot_{m}_by_{factor}.png")
            plt.savefig(boxplot_path)
            plt.close()
    
    # ------------------------------------------------------------------------------
    # 6. DEEPSEEK VS. EACH OTHER MODEL ANALYSIS
    # ------------------------------------------------------------------------------
    # This section compares the performance of 'deepseek' against each other model individually.
    deepseek_df = df[df["LLM"].str.lower() == "deepseek"]
    other_models = df[df["LLM"].str.lower() != "deepseek"]["LLM"].unique()
    
    with open(os.path.join(text_folder, "deepseek_analysis.txt"), "w") as f:
        f.write("Deepseek vs. Each Other Model Analysis:\n")
        for model in other_models:
            f.write(f"\nComparing deepseek with {model}:\n")
            model_df = df[df["LLM"] == model]
            for m in metrics:
                ds_mean = deepseek_df[m].mean()
                ds_std = deepseek_df[m].std()
                model_mean = model_df[m].mean()
                model_std = model_df[m].std()
                f.write(f"\nMetric: {m}\n")
                f.write(f"  Deepseek - Mean: {ds_mean:.2f}, Std: {ds_std:.2f}\n")
                f.write(f"  {model} - Mean: {model_mean:.2f}, Std: {model_std:.2f}\n")
                try:
                    # Perform an independent t-test assuming unequal variances
                    t_stat, p_val = ttest_ind(deepseek_df[m].dropna(), model_df[m].dropna(), equal_var=False)
                    f.write(f"  T-test: t = {t_stat:.2f}, p = {p_val:.4f}\n")
                except Exception as e:
                    f.write(f"  T-test could not be performed for {m} due to error: {e}\n")
    
# ------------------------------------------------------------------------------
# Run the analysis (ensure the CSV file "FinalResultsYesNo.csv" is in the working directory)
# ------------------------------------------------------------------------------
analyze_llm_results("FinalResultsYesNo.csv")
